# cap10_bc_sequence_risk_state_diagnostic_v1

Purpose: diagnose the remaining `wf_2022` drawdown gap after the cap10 joint-contract checkpoint selector chose BC initialization. The previous selector showed the BC path already clears the excess-return target (`+36.42pp`) but misses the MaxDD target (`18.46%` versus `<=15%`). This notebook rebuilds the same BC initialization, exports a daily path, and screens sequence-risk state features that could become the next target/reward input before any longer PPO run.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- Direct comparison baseline: `cap10_joint_contract_checkpoint_drift_selector_v1`.
- Role: `diagnose`.
- Changed layer: `model_selection` / `training_objective` / `risk_state_diagnostic` only.
- No actor is retained, promoted, or used for runtime execution from this notebook.
- Target gate remains excess >= `+30pp` and path MaxDD <= `15%` on fixed split plus walk-forward windows.

External basis:
- Qlib workflow separation and non-leaking validation: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO as a later policy-improvement candidate, not the first answer to BC drawdown drift: https://arxiv.org/abs/1707.06347
- Conservative / safe policy-improvement caution before policy updates: https://arxiv.org/abs/2006.04779 and https://arxiv.org/abs/1712.06924
- Backtest overfit / multiple-testing discipline: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import json
import math
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.path_diagnostics import (
    compare_interval_feature_deltas,
    find_worst_drawdown_intervals,
    make_backtest_daily_frame,
)
from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.settings import ACCOUNT, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'cap10_bc_sequence_risk_state_diagnostic_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_joint_contract_checkpoint_drift_selector_v1'
TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_foldlocal_joint_target_contract_v1'
TARGETS_PATH = TARGET_CARD / 'joint_target_contract.csv'
SOURCE_SELECTION_PATH = SOURCE_CARD / 'selection.json'
SOURCE_METRICS_PATH = SOURCE_CARD / 'checkpoint_candidate_metrics.csv'
FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'

SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
DRAWDOWN_BREACH = TARGET_MAX_DD / 100.0
PRE_BREACH_DAYS = 5
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)

WF_2022 = {
    'fold': 'wf_2022',
    'target_fold': 'wf_2022_train',
    'train': ('2010-03-11', '2020-12-31'),
    'valid': ('2021-01-01', '2021-12-31'),
    'test': ('2022-01-01', '2022-12-31'),
}

FEATURE_GROUPS = {
    'drawdown_path': ['drawdown_21', 'drawdown_63', 'cum_drop_20', 'cum_drop_30', 'days_since_high_63', 'drawdown_streak_63', 'below_ma63_days', 'below_anchor_ratio_63', 'new_low_ratio_21', 'rolling_new_low_ratio_63', 'loss_streak_21'],
    'volatility_stress': ['realized_vol_10', 'realized_vol_21', 'realized_vol_63', 'vol_of_vol_21', 'downside_semivol_21', 'vol_coef_21', 'vol_stress_score', 'jump_variation_5', 'realized_kurtosis_21'],
    'vix_term_stress': ['vix_level', 'vix_zscore', 'vix_percentile_252', 'vix_momentum_10', 'vvix_zscore_252', 'vix_term_slope', 'vix_short_ratio', 'vix_term_slope_velocity_5'],
    'trend_break': ['anchor_gap_180', 'anchor_slope_180', 'trend_gap_21', 'trend_gap_63', 'trend_slope_21', 'trend_slope_63', 'mom_21', 'mom_63', 'rsi_14', 'rsi_6', 'relative_strength_21'],
    'tracking_decay': ['tracking_error_5', 'tracking_error_21', 'tracking_error_vol_21', 'decay_63'],
    'liquidity_volume': ['intraday_range', 'volume_shock', 'relative_volume_zscore', 'volume_shock_velocity_5'],
}

CARD.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n')


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def pct_return(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors='coerce').dropna()
    if len(values) < 2:
        return 0.0
    return float((values.iloc[-1] / values.iloc[0] - 1.0) * 100.0)

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/.openclaw/workspace
card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/cap10_bc_sequence_risk_state_diagnostic_v1


In [2]:
source_selection = json.loads(SOURCE_SELECTION_PATH.read_text())
source_metrics = pd.read_csv(SOURCE_METRICS_PATH)
source_bc_row = source_metrics[
    source_metrics['fold'].eq(WF_2022['fold'])
    & source_metrics['candidate'].eq('bc_initialization')
].iloc[0].to_dict()

profile = load_profile(PROFILE_NAME)
frame = pd.read_pickle(FRAME_PATH).sort_index()
targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
    if col in targets.columns:
        targets[col] = targets[col].fillna(False).astype(bool)

train_start, train_end = map(pd.Timestamp, WF_2022['train'])
test_start, test_end = map(pd.Timestamp, WF_2022['test'])
valid_start, valid_end = map(pd.Timestamp, WF_2022['valid'])
fold_targets = targets[
    targets['fold'].eq(WF_2022['target_fold'])
    & targets['datetime'].between(train_start, train_end)
].copy()
leakage_rows = targets[
    targets['fold'].eq(WF_2022['target_fold'])
    & targets['datetime'].between(test_start, test_end)
]
valid_overlap = targets[
    targets['fold'].eq(WF_2022['target_fold'])
    & targets['datetime'].between(valid_start, valid_end)
]

bc_dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile, sample_size=0)
coverage = pd.DataFrame([
    {
        'experiment': EXPERIMENT,
        'source_decision': source_selection.get('decision'),
        'fold': WF_2022['fold'],
        'target_fold': WF_2022['target_fold'],
        'profile': PROFILE_NAME,
        'train_start': WF_2022['train'][0],
        'train_end': WF_2022['train'][1],
        'valid_start': WF_2022['valid'][0],
        'valid_end': WF_2022['valid'][1],
        'test_start': WF_2022['test'][0],
        'test_end': WF_2022['test'][1],
        'train_target_rows': int(len(fold_targets)),
        'bc_dataset_samples': int(len(bc_dataset.actions)),
        'test_target_overlap_rows': int(len(leakage_rows)),
        'valid_target_overlap_rows': int(len(valid_overlap)),
        'release_gate_rows': int(fold_targets.get('release_gate_state', pd.Series(False, index=fold_targets.index)).sum()),
        'downtrend_rows': int(fold_targets.get('is_downtrend_state', pd.Series(False, index=fold_targets.index)).sum()),
        'drawdown_control_rows': int(fold_targets.get('drawdown_control_state', pd.Series(False, index=fold_targets.index)).sum()),
        'source_bc_relative_return': float(source_bc_row['test_relative_return']),
        'source_bc_max_dd': float(source_bc_row['test_max_dd']),
        'source_bc_avg_pos': float(source_bc_row['test_avg_pos']),
    }
])
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {'rows': coverage.to_dict('records')})
display(coverage)


,experiment,source_decision,fold,target_fold,profile,train_start,train_end,valid_start,valid_end,test_start,...,train_target_rows,bc_dataset_samples,test_target_overlap_rows,valid_target_overlap_rows,release_gate_rows,downtrend_rows,drawdown_control_rows,source_bc_relative_return,source_bc_max_dd,source_bc_avg_pos
0,cap10_bc_sequence_risk_state_diagnostic_v1,checkpoint_drift_selector_partial_pass_wf2022_...,wf_2022,wf_2022_train,sparse_release_bc_ppo_screen_v1,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,...,253,253,0,252,40,118,49,36.419124,18.455088,0.280667


In [3]:
set_seed(SEED)
init_qlib(QLIB_DATA_PATH)

bc_fit = fit_state_objective_bc_actor_from_frame(
    frame,
    targets=fold_targets,
    profile=profile,
    train_config=BC_TRAIN_CONFIG,
    sample_weight_config=None,
    sample_size=0,
)
bc_slice_metrics = evaluate_bc_actor_slice_fit(
    bc_fit.actor,
    bc_fit.obs,
    bc_fit.actions,
    bc_dataset.metadata,
    slice_cols=('release_gate_state', 'is_downtrend_state', 'drawdown_control_state', 'is_rebound_cluster'),
)
bc_slice_metrics.to_csv(CARD / 'bc_slice_metrics.csv', index=False)

wf_dataset = build_dataset(
    train_seg=WF_2022['train'],
    valid_seg=WF_2022['valid'],
    test_seg=WF_2022['test'],
    feature_packs=tuple(profile.feature.packs),
    profile_name=profile.name,
)
test_df = segment_frame(wf_dataset, 'test')
price_frame = wf_dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
test_price = price_frame.loc[WF_2022['test'][0]:WF_2022['test'][1]]
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)

bt = backtest_profile(
    bc_fit.actor,
    test_df,
    test_price,
    position_cap_config=profile.position_cap,
    position_budget_config=profile.position_budget,
    execution_config=profile.execution,
    asset_idx=0,
    feature_columns=feature_columns,
    model_config=profile.model,
    mode_router_config=profile.mode_router,
    test_seg=WF_2022['test'],
)

feature_frame = frame[[c for c in frame.columns if c in set(sum(FEATURE_GROUPS.values(), []))]].copy()
daily = make_backtest_daily_frame(bt, label='bc_initialization_wf_2022', feature_frame=feature_frame)
daily['datetime'] = pd.to_datetime(daily['datetime'])
prices = test_price.reindex(pd.DatetimeIndex(daily['datetime'])).ffill().bfill().to_numpy(dtype=float)
initial_price = float(prices[0]) if len(prices) else 1.0
daily['price'] = prices
daily['strategy_nav'] = daily['account'].astype(float)
daily['buyhold_nav'] = ACCOUNT * (prices / initial_price)
daily['strategy_return_pct'] = daily['strategy_nav'] / float(daily['strategy_nav'].iloc[0]) * 100.0 - 100.0
daily['buyhold_return_pct'] = daily['buyhold_nav'] / float(daily['buyhold_nav'].iloc[0]) * 100.0 - 100.0
daily['excess_pct'] = daily['strategy_return_pct'] - daily['buyhold_return_pct']
daily['drawdown_pct'] = daily['drawdown'].astype(float) * 100.0
daily['position_change'] = daily['position'].astype(float).diff().fillna(0.0)
daily['next_strategy_return'] = daily['daily_return'].shift(-1)
daily['is_drawdown_breach_15pct'] = daily['drawdown'].astype(float) >= DRAWDOWN_BREACH

metrics = daily_path_metrics(daily[['datetime', 'strategy_nav', 'buyhold_nav', 'position']].copy())
metrics['source_bc_relative_return'] = float(source_bc_row['test_relative_return'])
metrics['source_bc_max_dd'] = float(source_bc_row['test_max_dd'])
metrics['source_bc_avg_pos'] = float(source_bc_row['test_avg_pos'])
metrics['relative_return_diff_vs_source'] = float(metrics['relative_return']) - float(source_bc_row['test_relative_return'])
metrics['max_dd_diff_vs_source'] = float(metrics['max_dd']) - float(source_bc_row['test_max_dd'])
metrics['avg_pos_diff_vs_source'] = float(metrics['avg_pos']) - float(source_bc_row['test_avg_pos'])
metrics['target_relative_return_pp'] = TARGET_RELATIVE_RETURN
metrics['target_max_dd_pct'] = TARGET_MAX_DD
metrics['target_gate'] = bool(float(metrics['relative_return']) >= TARGET_RELATIVE_RETURN and float(metrics['max_dd']) <= TARGET_MAX_DD)

# Put the path columns first, then feature columns.
front = [
    'datetime', 'label', 'account', 'strategy_nav', 'buyhold_nav', 'price', 'position',
    'nav', 'peak', 'drawdown', 'drawdown_pct', 'daily_return', 'strategy_return_pct',
    'buyhold_return_pct', 'excess_pct', 'position_change', 'next_strategy_return',
    'is_drawdown_breach_15pct',
]
ordered = [c for c in front if c in daily.columns] + [c for c in daily.columns if c not in front]
daily = daily[ordered]
daily.to_csv(CARD / 'bc_wf2022_daily_path.csv', index=False)
write_json(CARD / 'bc_wf2022_metrics.json', metrics)

display(pd.DataFrame([metrics]))
display(bc_slice_metrics)


[48689:MainThread](2026-05-26 07:59:28,156) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[48689:MainThread](2026-05-26 07:59:28,800) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[48689:MainThread](2026-05-26 07:59:28,800) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[48689:MainThread](2026-05-26 07:59:37,225) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,rows,start,end,return,buy_and_hold_return,relative_return,max_dd,avg_pos,source_bc_relative_return,source_bc_max_dd,source_bc_avg_pos,relative_return_diff_vs_source,max_dd_diff_vs_source,avg_pos_diff_vs_source,target_relative_return_pp,target_max_dd_pct,target_gate
0,250,2022-01-03,2022-12-29,-16.534504,-86.588072,70.053568,18.396429,0.14901,36.419124,18.455088,0.280667,33.634444,-0.058659,-0.131657,30.0,15.0,False


,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share
0,all,253,0.833992,1.073453,5,4,4,0.513834,4,0.616601
1,drawdown_control_state,49,0.918367,0.985184,3,2,4,0.857143,4,0.918367
2,is_downtrend_state,118,0.779661,1.129647,4,3,4,0.372881,4,0.533898
3,is_rebound_cluster,66,0.803030,1.101154,2,1,4,0.803030,4,1.000000
4,release_gate_state,40,1.000000,0.905056,1,1,4,1.000000,4,1.000000


In [4]:
def contiguous_windows(mask: pd.Series, *, label: str, source_daily: pd.DataFrame) -> pd.DataFrame:
    source = source_daily.copy().reset_index(drop=True)
    mask = pd.Series(mask.to_numpy(dtype=bool), index=source.index)
    rows: list[dict[str, Any]] = []
    start_idx: int | None = None
    for idx, active in enumerate(mask.tolist() + [False]):
        if active and start_idx is None:
            start_idx = idx
        elif not active and start_idx is not None:
            end_idx = idx - 1
            window = source.iloc[start_idx : end_idx + 1]
            if not window.empty:
                rows.append({
                    'window_type': label,
                    'start': str(pd.Timestamp(window['datetime'].iloc[0]).date()),
                    'end': str(pd.Timestamp(window['datetime'].iloc[-1]).date()),
                    'rows': int(len(window)),
                    'max_dd': float(window['drawdown_pct'].max()),
                    'end_drawdown': float(window['drawdown_pct'].iloc[-1]),
                    'return': pct_return(window['account']),
                    'buyhold_return': pct_return(window['buyhold_nav']),
                    'relative_return': pct_return(window['account']) - pct_return(window['buyhold_nav']),
                    'avg_pos': float(window['position'].mean()),
                    'min_daily_return': float(window['daily_return'].min()),
                    'max_position': float(window['position'].max()),
                })
            start_idx = None
    return pd.DataFrame(rows)

# Peak-to-trough intervals identify what actually caused the largest path loss.
worst_intervals = find_worst_drawdown_intervals(daily, top_n=5, min_depth=0.03)
worst_intervals['window_type'] = 'peak_to_trough'

breach_mask = daily['drawdown'].astype(float) >= DRAWDOWN_BREACH
breach_windows = contiguous_windows(breach_mask, label='above_15pct_drawdown', source_daily=daily)

pre_breach_mask = pd.Series(False, index=daily.index)
if not breach_windows.empty:
    for row in breach_windows.itertuples(index=False):
        start_ts = pd.Timestamp(row.start)
        candidates = daily[daily['datetime'] < start_ts].tail(PRE_BREACH_DAYS).index
        pre_breach_mask.loc[candidates] = True
pre_breach_windows = contiguous_windows(pre_breach_mask, label=f'pre_breach_{PRE_BREACH_DAYS}d', source_daily=daily)

worsening_mask = (daily['drawdown'].astype(float) >= 0.10) & (daily['drawdown'].astype(float).diff().fillna(0.0) > 0.0)
worsening_windows = contiguous_windows(worsening_mask, label='drawdown_worsening_above_10pct', source_daily=daily)

window_rows: list[pd.DataFrame] = []
if not worst_intervals.empty:
    interval_renamed = worst_intervals.rename(columns={'peak_date': 'peak_date', 'trough_date': 'trough_date'}).copy()
    interval_renamed['buyhold_return'] = np.nan
    interval_renamed['relative_return'] = np.nan
    interval_renamed['end_drawdown'] = interval_renamed['max_dd']
    interval_renamed['min_daily_return'] = np.nan
    interval_renamed['max_position'] = np.nan
    window_rows.append(interval_renamed)
for item in (breach_windows, pre_breach_windows, worsening_windows):
    if not item.empty:
        window_rows.append(item)

drawdown_windows = pd.concat(window_rows, ignore_index=True, sort=False) if window_rows else pd.DataFrame()
first_cols = ['window_type', 'rank', 'peak_date', 'trough_date', 'start', 'end', 'rows', 'max_dd', 'end_drawdown', 'return', 'buyhold_return', 'relative_return', 'avg_pos', 'min_daily_return', 'max_position']
drawdown_windows = drawdown_windows[[c for c in first_cols if c in drawdown_windows.columns] + [c for c in drawdown_windows.columns if c not in first_cols]]
drawdown_windows.to_csv(CARD / 'drawdown_breach_windows.csv', index=False)

display(drawdown_windows)


,window_type,rank,peak_date,trough_date,start,end,rows,max_dd,end_drawdown,return,buyhold_return,relative_return,avg_pos,min_daily_return,max_position,max_dd_fraction,peak_nav,trough_nav
0,peak_to_trough,1.0,2022-03-30,2022-11-03,2022-03-30,2022-11-03,152,18.396400,18.396400,-18.396400,NaN,NaN,0.179106,NaN,NaN,0.183964,1.003797,0.819134
1,above_15pct_drawdown,NaN,NaN,NaN,2022-10-10,2022-10-27,14,17.138012,16.960352,-0.799939,2.898560,-3.698499,0.165549,-0.024961,0.165549,NaN,NaN,NaN
2,above_15pct_drawdown,NaN,NaN,NaN,2022-10-31,2022-11-07,6,18.396429,15.635910,0.495416,7.511212,-7.015796,0.165549,-0.025903,0.165549,NaN,NaN,NaN
3,above_15pct_drawdown,NaN,NaN,NaN,2022-11-09,2022-11-09,1,16.204253,16.204253,0.000000,0.000000,0.000000,0.165549,-0.017703,0.165549,NaN,NaN,NaN
4,above_15pct_drawdown,NaN,NaN,NaN,2022-11-29,2022-11-29,1,15.070816,15.070816,0.000000,0.000000,0.000000,0.165549,-0.032242,0.165549,NaN,NaN,NaN
5,above_15pct_drawdown,NaN,NaN,NaN,2022-12-06,2022-12-06,1,15.096915,15.096915,0.000000,0.000000,0.000000,0.165549,-0.026049,0.165549,NaN,NaN,NaN
6,above_15pct_drawdown,NaN,NaN,NaN,2022-12-19,2022-12-20,2,15.759152,15.759152,-0.487672,-2.619268,2.131596,0.165549,-0.006301,0.165549,NaN,NaN,NaN
7,above_15pct_drawdown,NaN,NaN,NaN,2022-12-22,2022-12-29,5,18.044476,16.850209,-0.194670,-1.124741,0.930071,0.165549,-0.023503,0.165549,NaN,NaN,NaN
8,pre_breach_5d,NaN,NaN,NaN,2022-10-03,2022-10-07,5,14.147774,14.147774,-0.890372,-5.051544,4.161172,0.165549,-0.035030,0.165549,NaN,NaN,NaN
9,pre_breach_5d,NaN,NaN,NaN,2022-10-24,2022-10-28,5,16.960352,14.734405,1.554636,9.447001,-7.892365,0.165549,-0.013226,0.165549,NaN,NaN,NaN


In [5]:
candidate_features = [col for group in FEATURE_GROUPS.values() for col in group if col in daily.columns]
interval_deltas = compare_interval_feature_deltas(
    candidate_daily=daily,
    intervals=worst_intervals.head(3) if not worst_intervals.empty else worst_intervals,
    feature_cols=candidate_features,
)
if not interval_deltas.empty:
    interval_deltas.insert(0, 'source_type', 'peak_to_trough_interval')
    interval_deltas.insert(1, 'state_label', interval_deltas['interval_rank'].map(lambda rank: f'interval_{int(rank)}'))

sequence_mask = breach_mask | pre_breach_mask | worsening_mask
state_masks = {
    'breach_or_pre_breach_or_worsening': sequence_mask,
    'above_15pct_drawdown': breach_mask,
    f'pre_breach_{PRE_BREACH_DAYS}d': pre_breach_mask,
    'drawdown_worsening_above_10pct': worsening_mask,
}

state_rows: list[dict[str, Any]] = []
for state_label, mask in state_masks.items():
    mask = pd.Series(mask.to_numpy(dtype=bool), index=daily.index)
    state = daily.loc[mask]
    reference = daily.loc[~mask]
    if state.empty or reference.empty:
        continue
    for feature in candidate_features:
        state_series = pd.to_numeric(state[feature], errors='coerce').dropna()
        ref_series = pd.to_numeric(reference[feature], errors='coerce').dropna()
        all_series = pd.to_numeric(daily[feature], errors='coerce').dropna()
        if state_series.empty or ref_series.empty or all_series.empty:
            continue
        ref_std = float(ref_series.std(ddof=0))
        if ref_std == 0.0 or not math.isfinite(ref_std):
            ref_std = float(all_series.std(ddof=0))
        if ref_std == 0.0 or not math.isfinite(ref_std):
            continue
        state_mean = float(state_series.mean())
        ref_mean = float(ref_series.mean())
        delta = state_mean - ref_mean
        group = next((name for name, cols in FEATURE_GROUPS.items() if feature in cols), 'other')
        state_rows.append({
            'source_type': 'state_mask',
            'state_label': state_label,
            'feature_group': group,
            'feature': feature,
            'state_rows': int(len(state)),
            'reference_rows': int(len(reference)),
            'state_mean': state_mean,
            'reference_mean': ref_mean,
            'delta': delta,
            'delta_z': delta / ref_std,
            'abs_delta_z': abs(delta / ref_std),
            'state_coverage': float(len(state_series) / max(len(state), 1)),
            'reference_coverage': float(len(ref_series) / max(len(reference), 1)),
            'state_avg_pos': float(state['position'].mean()),
            'reference_avg_pos': float(reference['position'].mean()),
            'state_avg_drawdown': float(state['drawdown_pct'].mean()),
            'reference_avg_drawdown': float(reference['drawdown_pct'].mean()),
            'state_avg_next_return': float(pd.to_numeric(state['next_strategy_return'], errors='coerce').mean()),
            'reference_avg_next_return': float(pd.to_numeric(reference['next_strategy_return'], errors='coerce').mean()),
        })
state_deltas = pd.DataFrame(state_rows)
if not state_deltas.empty:
    state_deltas = state_deltas.sort_values(['state_label', 'abs_delta_z', 'feature'], ascending=[True, False, True]).reset_index(drop=True)

feature_differences = pd.concat(
    [df for df in (interval_deltas, state_deltas) if df is not None and not df.empty],
    ignore_index=True,
    sort=False,
) if (not interval_deltas.empty or not state_deltas.empty) else pd.DataFrame()
feature_differences.to_csv(CARD / 'sequence_risk_feature_differences.csv', index=False)

display(feature_differences.head(40))


,source_type,state_label,interval_rank,interval_start,interval_end,feature,interval_mean,reference_mean,delta,delta_z,...,state_rows,reference_rows,state_mean,state_coverage,state_avg_pos,reference_avg_pos,state_avg_drawdown,reference_avg_drawdown,state_avg_next_return,reference_avg_next_return
0,peak_to_trough_interval,interval_1,1.0,2022-03-30,2022-11-03,anchor_gap_180,-0.538761,-0.184071,-0.354689,-1.561590,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,peak_to_trough_interval,interval_1,1.0,2022-03-30,2022-11-03,below_ma63_days,0.659983,0.242630,0.417353,1.485600,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,peak_to_trough_interval,interval_1,1.0,2022-03-30,2022-11-03,tracking_error_vol_21,0.076507,0.065631,0.010876,1.080772,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,peak_to_trough_interval,interval_1,1.0,2022-03-30,2022-11-03,below_anchor_ratio_63,0.983500,0.570457,0.413044,1.068589,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,peak_to_trough_interval,interval_1,1.0,2022-03-30,2022-11-03,rolling_new_low_ratio_63,0.148392,0.091351,0.057041,1.002950,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,peak_to_trough_interval,interval_1,1.0,2022-03-30,2022-11-03,drawdown_63,-0.497821,-0.342284,-0.155537,-0.993228,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,peak_to_trough_interval,interval_1,1.0,2022-03-30,2022-11-03,drawdown_streak_63,1.000000,0.722870,0.277130,0.900287,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,peak_to_trough_interval,interval_1,1.0,2022-03-30,2022-11-03,vvix_zscore_252,-1.136533,-0.009902,-1.126631,-0.882839,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,peak_to_trough_interval,interval_1,1.0,2022-03-30,2022-11-03,mom_63,-0.335226,-0.052007,-0.283219,-0.865499,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,peak_to_trough_interval,interval_1,1.0,2022-03-30,2022-11-03,new_low_ratio_21,0.230576,0.140914,0.089663,0.865491,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
def threshold_flag(series: pd.Series, *, direction: str) -> tuple[pd.Series, float]:
    values = pd.to_numeric(series, errors='coerce')
    if direction == 'high':
        threshold = float(values.quantile(0.75))
        flag = values >= threshold
    else:
        threshold = float(values.quantile(0.25))
        flag = values <= threshold
    return flag.fillna(False), threshold

primary_state = 'breach_or_pre_breach_or_worsening'
primary_deltas = state_deltas[state_deltas['state_label'].eq(primary_state)].copy() if not state_deltas.empty else pd.DataFrame()

screen_rows: list[dict[str, Any]] = []
for group, features in FEATURE_GROUPS.items():
    available = [feature for feature in features if feature in daily.columns]
    if not available or primary_deltas.empty:
        continue
    ranked = primary_deltas[primary_deltas['feature'].isin(available)].sort_values('abs_delta_z', ascending=False)
    if ranked.empty:
        continue
    top = ranked.head(3).copy()
    flags = []
    feature_specs = []
    for row in top.itertuples(index=False):
        direction = 'high' if float(row.delta_z) >= 0 else 'low'
        flag, threshold = threshold_flag(daily[str(row.feature)], direction=direction)
        flags.append(flag)
        feature_specs.append({
            'feature': str(row.feature),
            'direction': direction,
            'threshold': threshold,
            'delta_z': float(row.delta_z),
            'abs_delta_z': float(row.abs_delta_z),
        })
    if not flags:
        continue
    group_flag = pd.concat(flags, axis=1).any(axis=1)
    flag_rate = float(group_flag.mean())
    breach_capture = float(group_flag[breach_mask].mean()) if bool(breach_mask.any()) else 0.0
    pre_breach_capture = float(group_flag[pre_breach_mask].mean()) if bool(pre_breach_mask.any()) else 0.0
    sequence_capture = float(group_flag[sequence_mask].mean()) if bool(sequence_mask.any()) else 0.0
    non_sequence_rate = float(group_flag[~sequence_mask].mean()) if bool((~sequence_mask).any()) else 0.0
    flagged = daily.loc[group_flag]
    unflagged = daily.loc[~group_flag]
    mean_abs_delta_z = float(top['abs_delta_z'].mean())
    max_abs_delta_z = float(top['abs_delta_z'].max())
    specificity = max(0.0, sequence_capture - non_sequence_rate)
    score = (
        min(mean_abs_delta_z / 2.0, 1.5)
        + breach_capture
        + 0.50 * pre_breach_capture
        + 0.50 * specificity
        - 0.25 * flag_rate
    )
    candidate_gate = bool(
        max_abs_delta_z >= 0.75
        and breach_capture >= 0.60
        and sequence_capture >= 0.50
        and flag_rate <= 0.65
    )
    screen_rows.append({
        'feature_group': group,
        'candidate_gate': candidate_gate,
        'score': score,
        'mean_abs_delta_z': mean_abs_delta_z,
        'max_abs_delta_z': max_abs_delta_z,
        'flag_rate': flag_rate,
        'breach_capture': breach_capture,
        'pre_breach_capture': pre_breach_capture,
        'sequence_capture': sequence_capture,
        'non_sequence_rate': non_sequence_rate,
        'specificity': specificity,
        'flagged_rows': int(group_flag.sum()),
        'sequence_rows': int(sequence_mask.sum()),
        'breach_rows': int(breach_mask.sum()),
        'pre_breach_rows': int(pre_breach_mask.sum()),
        'flagged_avg_pos': float(flagged['position'].mean()) if not flagged.empty else 0.0,
        'unflagged_avg_pos': float(unflagged['position'].mean()) if not unflagged.empty else 0.0,
        'flagged_avg_drawdown_pct': float(flagged['drawdown_pct'].mean()) if not flagged.empty else 0.0,
        'unflagged_avg_drawdown_pct': float(unflagged['drawdown_pct'].mean()) if not unflagged.empty else 0.0,
        'flagged_next_return_mean': float(pd.to_numeric(flagged['next_strategy_return'], errors='coerce').mean()) if not flagged.empty else 0.0,
        'unflagged_next_return_mean': float(pd.to_numeric(unflagged['next_strategy_return'], errors='coerce').mean()) if not unflagged.empty else 0.0,
        'feature_specs': json.dumps(json_safe(feature_specs), sort_keys=True),
    })

candidate_screen = pd.DataFrame(screen_rows)
if not candidate_screen.empty:
    candidate_screen = candidate_screen.sort_values(['candidate_gate', 'score', 'mean_abs_delta_z'], ascending=[False, False, False]).reset_index(drop=True)
candidate_screen.to_csv(CARD / 'risk_state_candidate_screen.csv', index=False)

selected_candidate = candidate_screen.iloc[0].to_dict() if not candidate_screen.empty else {}
any_candidate_gate = bool(not candidate_screen.empty and candidate_screen['candidate_gate'].astype(bool).any())

if any_candidate_gate:
    decision = 'sequence_risk_state_candidate_ready_for_target_screen_no_promotion'
    next_mechanism = 'add_sequence_risk_state_cap_to_joint_contract_then_screen_bc_and_checkpoint_drift'
else:
    decision = 'sequence_risk_feature_screen_inconclusive_no_promotion'
    next_mechanism = 'aggregate_more_failed_daily_paths_before_new_target_or_reward_design'

display(candidate_screen)


,feature_group,candidate_gate,score,mean_abs_delta_z,max_abs_delta_z,flag_rate,breach_capture,pre_breach_capture,sequence_capture,non_sequence_rate,...,sequence_rows,breach_rows,pre_breach_rows,flagged_avg_pos,unflagged_avg_pos,flagged_avg_drawdown_pct,unflagged_avg_drawdown_pct,flagged_next_return_mean,unflagged_next_return_mean,feature_specs
0,trend_break,True,2.092371,0.881028,1.908566,0.564,1.000000,1.000000,0.985714,0.400000,...,70,30,33,0.166073,0.126938,10.188497,1.853297,-0.000467,-0.000874,"[{""abs_delta_z"": 1.9085659879088879, ""delta_z""..."
1,tracking_decay,True,1.961850,0.816242,1.347655,0.548,0.966667,0.939394,0.914286,0.405556,...,70,30,33,0.156633,0.139768,9.528942,2.947986,-0.000203,-0.001178,"[{""abs_delta_z"": 1.3476549592424019, ""delta_z""..."
2,volatility_stress,True,1.864461,1.239387,1.718616,0.528,0.733333,0.909091,0.800000,0.422222,...,70,30,33,0.174978,0.119961,8.399095,4.490737,-0.000324,-0.001001,"[{""abs_delta_z"": 1.7186155455380494, ""delta_z""..."
3,drawdown_path,False,1.806186,0.711928,0.818191,0.788,1.000000,1.000000,1.000000,0.705556,...,70,30,33,0.162671,0.098230,8.134869,0.679589,-0.000653,-0.000618,"[{""abs_delta_z"": 0.8181913852925198, ""delta_z""..."
4,vix_term_stress,False,1.289004,0.701812,0.983698,0.444,0.566667,0.787879,0.571429,0.394444,...,70,30,33,0.151695,0.146865,8.307754,5.154149,-0.001126,-0.000265,"[{""abs_delta_z"": 0.9836984259604697, ""delta_z""..."
5,liquidity_volume,False,0.769586,0.182514,0.234516,0.404,0.500000,0.484848,0.457143,0.383333,...,70,30,33,0.157364,0.143347,7.169045,6.137677,-0.000765,-0.000565,"[{""abs_delta_z"": 0.23451550450006545, ""delta_z..."


In [7]:
sidecars = [
    'coverage.csv',
    'coverage.json',
    'bc_slice_metrics.csv',
    'bc_wf2022_daily_path.csv',
    'bc_wf2022_metrics.json',
    'drawdown_breach_windows.csv',
    'sequence_risk_feature_differences.csv',
    'risk_state_candidate_screen.csv',
    'selection.json',
    'summary.json',
]

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'cap10_joint_contract_checkpoint_drift_selector_v1',
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'fold': WF_2022['fold'],
    'changed_layer': ['model_selection', 'training_objective', 'risk_state_diagnostic'],
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate the BC diagnostic path',
    'promotion_blocker': 'This is a single-fold sequence-risk diagnostic. Promotion still requires fixed-split plus wf_2020/wf_2022/wf_2023_2024 validation with excess >= +30pp and MaxDD <= 15%.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'bc_relative_return': float(metrics['relative_return']),
    'bc_max_dd': float(metrics['max_dd']),
    'bc_avg_pos': float(metrics['avg_pos']),
    'bc_target_gate': bool(metrics['target_gate']),
    'drawdown_gap_pp': max(0.0, float(metrics['max_dd']) - TARGET_MAX_DD),
    'breach_rows': int(breach_mask.sum()),
    'pre_breach_rows': int(pre_breach_mask.sum()),
    'sequence_rows': int(sequence_mask.sum()),
    'candidate_gate_count': int(candidate_screen['candidate_gate'].sum()) if not candidate_screen.empty else 0,
    'selected_feature_group': selected_candidate.get('feature_group'),
    'selected_candidate_score': selected_candidate.get('score'),
    'selected_breach_capture': selected_candidate.get('breach_capture'),
    'selected_flag_rate': selected_candidate.get('flag_rate'),
    'sidecars': sidecars,
    'source_decision': source_selection.get('decision'),
    'external_sources': source_selection.get('external_sources', [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://arxiv.org/abs/1707.06347',
        'https://arxiv.org/abs/2006.04779',
        'https://arxiv.org/abs/1712.06924',
        'https://doi.org/10.21314/JCF.2016.322',
    ]),
}

summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'coverage': coverage.to_dict('records'),
    'bc_fit_metrics': bc_fit.metrics,
    'bc_backtest_metrics': metrics,
    'drawdown_windows': drawdown_windows.to_dict('records') if not drawdown_windows.empty else [],
    'top_sequence_risk_feature_differences': feature_differences.head(25).to_dict('records') if not feature_differences.empty else [],
    'risk_state_candidate_screen': candidate_screen.to_dict('records') if not candidate_screen.empty else [],
    'selected_candidate': selected_candidate,
    'risk_notes': [
        'The BC initialization is useful for excess return but fails the <=15% MaxDD gate on wf_2022.',
        'This notebook is diagnostic only and should not be used to promote or retain an actor.',
        'A passing sequence-risk candidate means the next valid step is a target/reward screen, not a longer PPO run by itself.',
    ],
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)

lines = [
    '## Conclusion',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
    f"Rebuilt `wf_2022` BC path: relative return `{metrics['relative_return']:.2f}pp`, MaxDD `{metrics['max_dd']:.2f}%`, avg position `{metrics['avg_pos']:.3f}`. The excess gate passes, but MaxDD misses the `15%` target by `{selection['drawdown_gap_pp']:.2f}pp`.",
    '',
    f"Drawdown breach rows: `{selection['breach_rows']}`; pre-breach rows: `{selection['pre_breach_rows']}`; sequence-risk rows: `{selection['sequence_rows']}`.",
    '',
]
if selected_candidate:
    lines.append(
        f"Top candidate group: `{selected_candidate['feature_group']}` with score `{selected_candidate['score']:.3f}`, breach capture `{selected_candidate['breach_capture']:.2%}`, sequence capture `{selected_candidate['sequence_capture']:.2%}`, and flag rate `{selected_candidate['flag_rate']:.2%}`."
    )
    lines.append('')
    lines.append(f"Candidate gate count: `{selection['candidate_gate_count']}`.")
else:
    lines.append('No candidate feature group had enough coverage to form a sequence-risk state.')
lines.extend([
    '',
    'Risk note: no runtime promotion, no retained actor, and no longer PPO continuation is allowed directly from this card.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `bc_wf2022_daily_path.csv`, `drawdown_breach_windows.csv`, `sequence_risk_feature_differences.csv`, `risk_state_candidate_screen.csv`, `selection.json`, `summary.json`.',
])
display(Markdown(chr(10).join(lines)))


## Conclusion

Decision: `sequence_risk_state_candidate_ready_for_target_screen_no_promotion`.

Rebuilt `wf_2022` BC path: relative return `70.05pp`, MaxDD `18.40%`, avg position `0.149`. The excess gate passes, but MaxDD misses the `15%` target by `3.40pp`.

Drawdown breach rows: `30`; pre-breach rows: `33`; sequence-risk rows: `70`.

Top candidate group: `trend_break` with score `2.092`, breach capture `100.00%`, sequence capture `98.57%`, and flag rate `56.40%`.

Candidate gate count: `3`.

Risk note: no runtime promotion, no retained actor, and no longer PPO continuation is allowed directly from this card.

Next action: `add_sequence_risk_state_cap_to_joint_contract_then_screen_bc_and_checkpoint_drift`.

Artifacts: `bc_wf2022_daily_path.csv`, `drawdown_breach_windows.csv`, `sequence_risk_feature_differences.csv`, `risk_state_candidate_screen.csv`, `selection.json`, `summary.json`.